# WELCOME TO CRESENCIO'S LAB 9 ADDITIONAL EXERCISES

In [1]:
# Importing necessary modules, with auto-install for missing packages.

import subprocess, sys
 
# Auto-install any missing packages (safe to re-run)
_required = ["requests", "beautifulsoup4", "lxml", "pandas", "matplotlib", "pillow"]
for _pkg in _required:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", _pkg, "-q"],
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
    )
 
import requests
from bs4 import BeautifulSoup
import pandas as pd
import json
import os
import re
import time
import csv
from datetime import datetime
import matplotlib
matplotlib.use("Agg")   # non-interactive backend -- works in any Jupyter
import matplotlib.pyplot as plt
import matplotlib.animation as animation
 
print("All modules installed and imported successfully!")
print(f"  Python   : {sys.version.split()[0]}")
print(f"  Pandas   : {pd.__version__}")
print(f"  Requests : {requests.__version__}")
 
 
# ─────────────────────────────────────────────────────────────────────────────
# SHARED HELPER — scrape_weather()
# Used by both Exercise 1 and Exercise 2.
# Source: timeanddate.com/weather/philippines/<city_slug>
# ─────────────────────────────────────────────────────────────────────────────
 
_HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0.0.0 Safari/537.36"
    ),
    "Accept-Language": "en-US,en;q=0.9",
}
 
 
def _clean(text):
    """Remove non-breaking spaces and collapse whitespace."""
    return re.sub(r"\s+", " ", text.replace("\xa0", " ")).strip()
 
 
def _table_value(soup, keyword):
    """
    Walk every table row; return the adjacent cell's text
    when the first cell contains keyword.
    """
    for row in soup.find_all("tr"):
        cells = row.find_all(["td", "th"])
        for i, cell in enumerate(cells):
            if keyword.lower() in cell.get_text().lower() and i + 1 < len(cells):
                return _clean(cells[i + 1].get_text())
    return "N/A"
 
 
def scrape_weather(city_slug):
    """
    Scrape current + extended forecast weather from timeanddate.com
    for a Philippine city.
 
    city_slug : URL slug, e.g. 'manila', 'cebu', 'davao'.
 
    Returns a dict with:
        temperature, feels_like, condition, temp_high, temp_low,
        wind, humidity, pressure, visibility, dew_point,
        forecast_rows (list of dicts: forecast_date, temp_hi, temp_lo,
                       condition, humidity)
    Returns None on network failure.
    """
    base_url     = f"https://www.timeanddate.com/weather/philippines/{city_slug}"
    forecast_url = f"{base_url}/ext"
 
    try:
        resp = requests.get(base_url, headers=_HEADERS, timeout=15)
        resp.raise_for_status()
    except requests.exceptions.RequestException as exc:
        print(f"  Could not reach {base_url}: {exc}")
        return None
 
    soup = BeautifulSoup(resp.text, "lxml")
 
    # Current temperature
    temp_el = soup.select_one("#qlook .h2") or soup.select_one(".h2")
    temperature = _clean(temp_el.get_text()) if temp_el else "N/A"
 
    # Weather condition
    cond_el = soup.select_one("#qlook p")
    condition = _clean(cond_el.get_text()) if cond_el else _table_value(soup, "Condition")
 
    # Forecast high / low
    hi_el = soup.select_one(".hi-lo .hi")
    lo_el = soup.select_one(".hi-lo .lo")
    temp_high = _clean(hi_el.get_text()) if hi_el else "N/A"
    temp_low  = _clean(lo_el.get_text()) if lo_el else "N/A"
 
    # Detailed attributes
    feels_like = _table_value(soup, "Feels Like")
    wind       = _table_value(soup, "Wind")
    humidity   = _table_value(soup, "Humidity")
    pressure   = _table_value(soup, "Pressure")
    visibility = _table_value(soup, "Visibility")
    dew_point  = _table_value(soup, "Dew Point")
 
    # Extended forecast page
    forecast_rows = []
    try:
        freq = requests.get(forecast_url, headers=_HEADERS, timeout=15)
        freq.raise_for_status()
        fsoup = BeautifulSoup(freq.text, "lxml")
        tbl = fsoup.select_one("#wt-ext") or fsoup.find("table")
        if tbl:
            for row in tbl.find_all("tr")[1:]:
                cols = row.find_all("td")
                if len(cols) >= 3:
                    forecast_rows.append({
                        "forecast_date": _clean(cols[0].get_text()),
                        "temp_hi":       _clean(cols[1].get_text()),
                        "temp_lo":       _clean(cols[2].get_text()),
                        "condition":     _clean(cols[3].get_text()) if len(cols) > 3 else "N/A",
                        "humidity":      _clean(cols[5].get_text()) if len(cols) > 5 else "N/A",
                    })
    except requests.exceptions.RequestException:
        pass   # forecast is optional
 
    return {
        "temperature":   temperature,
        "feels_like":    feels_like,
        "condition":     condition,
        "temp_high":     temp_high,
        "temp_low":      temp_low,
        "wind":          wind,
        "humidity":      humidity,
        "pressure":      pressure,
        "visibility":    visibility,
        "dew_point":     dew_point,
        "forecast_rows": forecast_rows,
    }
 
 

All modules installed and imported successfully!
  Python   : 3.14.3
  Pandas   : 3.0.2
  Requests : 2.33.1


**Additional Exercise #1: Daily Weather Logger Simulation**
Functionality: Build a command-line Python program that simulates daily logging of current and forecast weather data for a single Philippine city from timeanddate.com/weather philippines, including additional weather attributes such as humidity, pressure, visibility, and dew point. The script introduces time delays between runs to mimic scheduled data collection.

Input:

• City name (string), entered by the user interactively.

• Number of simulated days (integer), entered by the user; defaults to 3 if no valid input is provided.
 
Calculations:

• For each simulation run:
    
    • Scrape current weather data for the specified city, including:

    • Temperature
    
    • Feels like temperature

    • Weather condition (e.g., "Light rain. Overcast.")

    • Forecast high / low
    
    • Wind speed and direction
    
    • Humidity
    
    • Pressure

    • Visibility
    
    • Dew Point

    • Scrape forecast data for N days (1 ≤ N ≤ 7).

• Clean scraped text to remove non-breaking spaces and unwanted characters.

• Append all collected weather data to a CSV file with the following columns: timestamp, city, forecast_date, temperature, temperature_min, temperature_max, condition, wind, humidity,
pressure, visibility, dew_point, feels_like.

• Use time.sleep() to delay the next scraping run (e.g., 10 seconds).

• Write CSV headers only if the file does not already exist.

Output:

• A CSV file named weather_data.csv containing timestamped entries across simulated days,
including detailed current weather attributes for the selected city.

In [2]:
#  CELL 2 — Additional Exercise 1: Daily Weather Logger (CSV)            
#  INPUT GUIDE
#  -----------
#  "Enter Philippine city name"
#      Type the city slug as it appears in the timeanddate.com URL.
#      Valid examples:  manila  cebu  davao  quezon  makati  pasig
#                       taguig  iloilo  bacolod  zamboanga  baguio
#                       cagayan-de-oro  antipolo  angeles  pasay
#      Press ENTER (blank) to use the default: manila
#
#  "Number of simulated days to log"
#      Type a whole number, e.g.:  3
#      Press ENTER (blank) to use the default: 3
#      Each run fetches live data once then waits 10 seconds.
# ─────────────────────────────────────────────────────────────────────────────
 
def exercise1_daily_weather_logger():
    print("=" * 62)
    print("  EXERCISE 1 - Daily Weather Logger Simulation  ->  CSV")
    print("=" * 62)
 
    raw_city = input(
        "\nEnter Philippine city name (e.g. manila, cebu, davao)\n"
        "[press ENTER for default: manila] > "
    ).strip().lower().replace(" ", "-")
    city_slug = raw_city or "manila"
 
    try:
        days = int(
            input("Number of simulated days to log [press ENTER for default: 3] > ").strip()
        )
        if days < 1:
            raise ValueError
    except (ValueError, EOFError):
        days = 3
        print("  Using default: 3 days.")
 
    CSV_PATH = "weather_data.csv"
    FIELDNAMES = [
        "timestamp", "city", "forecast_date",
        "temperature", "temperature_min", "temperature_max",
        "condition", "wind", "humidity", "pressure",
        "visibility", "dew_point", "feels_like",
    ]
    file_existed = os.path.exists(CSV_PATH)
 
    print(f"\nScraping weather for '{city_slug}' - {days} simulated day(s)...\n")
 
    for run in range(1, days + 1):
        print(f"  -- Run {run}/{days} " + "-" * 40)
        weather = scrape_weather(city_slug)
 
        if weather is None:
            print("  Skipping run - fetch failed.")
        else:
            ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
            sources = weather["forecast_rows"][:7] if weather["forecast_rows"] else [None]
            rows_to_write = []
            for frow in sources:
                rows_to_write.append({
                    "timestamp":       ts,
                    "city":            city_slug,
                    "forecast_date":   frow["forecast_date"] if frow else ts[:10],
                    "temperature":     weather["temperature"],
                    "temperature_min": frow["temp_lo"]   if frow else weather["temp_low"],
                    "temperature_max": frow["temp_hi"]   if frow else weather["temp_high"],
                    "condition":       frow["condition"] if frow else weather["condition"],
                    "wind":            weather["wind"],
                    "humidity":        frow.get("humidity", weather["humidity"]) if frow else weather["humidity"],
                    "pressure":        weather["pressure"],
                    "visibility":      weather["visibility"],
                    "dew_point":       weather["dew_point"],
                    "feels_like":      weather["feels_like"],
                })
 
            with open(CSV_PATH, "a", newline="", encoding="utf-8") as f:
                writer = csv.DictWriter(f, fieldnames=FIELDNAMES)
                if not file_existed:
                    writer.writeheader()
                    file_existed = True
                writer.writerows(rows_to_write)
 
            print(f"  Logged {len(rows_to_write)} row(s)")
            print(f"  Current temp : {weather['temperature']}")
            print(f"  Condition    : {weather['condition'][:60]}")
            print(f"  Humidity     : {weather['humidity']}")
            print(f"  Wind         : {weather['wind']}")
 
        if run < days:
            print("  Waiting 10 seconds before next run...")
            time.sleep(10)
 
    print(f"\n{'─'*62}")
    print(f"Done! Data saved to '{CSV_PATH}'")
    df_check = pd.read_csv(CSV_PATH, encoding="utf-8")
    print(f"Total rows now in file: {len(df_check)}")
    print("\nLast 5 rows:")
    print(df_check.tail(5).to_string(index=False))
 
 
exercise1_daily_weather_logger()

  EXERCISE 1 - Daily Weather Logger Simulation  ->  CSV
  Using default: 3 days.

Scraping weather for 'manila' - 3 simulated day(s)...

  -- Run 1/3 ----------------------------------------
  Logged 7 row(s)
  Current temp : 34 °C
  Condition    : Partly sunny.
  Humidity     : 47%
  Wind         : 21 km/h
  Waiting 10 seconds before next run...
  -- Run 2/3 ----------------------------------------
  Logged 7 row(s)
  Current temp : 34 °C
  Condition    : Partly sunny.
  Humidity     : 47%
  Wind         : 21 km/h
  Waiting 10 seconds before next run...
  -- Run 3/3 ----------------------------------------
  Logged 7 row(s)
  Current temp : 34 °C
  Condition    : Partly sunny.
  Humidity     : 47%
  Wind         : 21 km/h

──────────────────────────────────────────────────────────────
Done! Data saved to 'weather_data.csv'
Total rows now in file: 102

Last 5 rows:
          timestamp   city forecast_date temperature                 temperature_min temperature_max condition    wind hum

**Additional Exercise #2: JSON Weather Exporter**
Functionality: Build a command-line Python program that scrapes current and forecast weather data for any Philippine city from timeanddate.com/weather/philippines, including extended attributes such as humidity, pressure, visibility, and dew point, and exports the cleaned data into a structured JSON file.

Input:

• City name (string), entered by the user interactively.

• Number of forecast days (integer, optional), entered by the user; defaults to 3 if no valid input is provided.

Calculations:

• Scrape current weather data for the specified city, including:

• Temperature

• Feels like temperature

• Weather condition (e.g., "Light rain. Overcast.")

• Forecast high / low

• Wind speed and direction

• Humidity

• Pressure

• Visibility

• Dew Point

• Scrape forecast data for N days (1 ≤ N ≤ 7).

• Clean scraped text to remove non-breaking spaces and unwanted characters.

• Structure the weather data into a list of dictionaries using the following fields: timestamp, city, forecast_date, temperature, temperature_min, temperature_max, condition, wind,
humidity, pressure, visibility, dew_point, feels_like.

• Append the new entries to a JSON file, creating the file if it does not already exist.

Output:

• A JSON file named weather_data.json containing a structured list of weather records with all major attributes in clean, timestamped format.

In [3]:
#  CELL 3 — Additional Exercise 2: JSON Weather Exporter                
#
#  INPUT GUIDE
#  -----------
#  "Enter Philippine city name"
#      Same city slugs as Exercise 1 (e.g. manila, cebu, davao)
#      Press ENTER (blank) for default: manila
#
#  "Number of forecast days to export (1-7)"
#      Whole number between 1 and 7, e.g.:  5
#      Press ENTER (blank) for default: 3
# ─────────────────────────────────────────────────────────────────────────────
 
def exercise2_json_weather_exporter():
    print("=" * 62)
    print("  EXERCISE 2 - JSON Weather Exporter")
    print("=" * 62)
 
    raw_city = input(
        "\nEnter Philippine city name (e.g. manila, cebu, davao)\n"
        "[press ENTER for default: manila] > "
    ).strip().lower().replace(" ", "-")
    city_slug = raw_city or "manila"
 
    try:
        n_days = int(
            input("Number of forecast days to export (1-7) [press ENTER for default: 3] > ").strip()
        )
        n_days = max(1, min(7, n_days))
    except (ValueError, EOFError):
        n_days = 3
        print("  Using default: 3 days.")
 
    JSON_PATH = "weather_data.json"
 
    print(f"\nScraping weather for '{city_slug}'...")
    weather = scrape_weather(city_slug)
 
    if weather is None:
        print("Failed to retrieve weather data.")
        print("Check: (1) the city slug spelling, (2) internet connection.")
        return
 
    ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    sources = weather["forecast_rows"][:n_days] if weather["forecast_rows"] else [None]
    new_records = []
    for frow in sources:
        new_records.append({
            "timestamp":       ts,
            "city":            city_slug,
            "forecast_date":   frow["forecast_date"] if frow else ts[:10],
            "temperature":     weather["temperature"],
            "temperature_min": frow["temp_lo"]   if frow else weather["temp_low"],
            "temperature_max": frow["temp_hi"]   if frow else weather["temp_high"],
            "condition":       frow["condition"] if frow else weather["condition"],
            "wind":            weather["wind"],
            "humidity":        frow.get("humidity", weather["humidity"]) if frow else weather["humidity"],
            "pressure":        weather["pressure"],
            "visibility":      weather["visibility"],
            "dew_point":       weather["dew_point"],
            "feels_like":      weather["feels_like"],
        })
 
    # Append to existing file or create new
    existing = []
    if os.path.exists(JSON_PATH):
        try:
            with open(JSON_PATH, "r", encoding="utf-8") as f:
                existing = json.load(f)
            if not isinstance(existing, list):
                existing = []
        except (json.JSONDecodeError, ValueError):
            existing = []
 
    existing.extend(new_records)
 
    with open(JSON_PATH, "w", encoding="utf-8") as f:
        json.dump(existing, f, ensure_ascii=False, indent=4)
 
    print(f"\nExported {len(new_records)} record(s) to '{JSON_PATH}'")
    print(f"Total records in file: {len(existing)}")
    print("\nSample record (first exported):")
    print(json.dumps(new_records[0], indent=4, ensure_ascii=False))
 
 
exercise2_json_weather_exporter()

  EXERCISE 2 - JSON Weather Exporter


  Using default: 3 days.

Scraping weather for 'manila'...

Exported 3 record(s) to 'weather_data.json'
Total records in file: 27

Sample record (first exported):
{
    "timestamp": "2026-04-27 12:47:10",
    "city": "manila",
    "forecast_date": "",
    "temperature": "34 °C",
    "temperature_min": "Isolated tstorms late. Broken clouds.",
    "temperature_max": "36 / 27 °C",
    "condition": "41 °C",
    "wind": "21 km/h",
    "humidity": "↑",
    "pressure": "1010 mbar",
    "visibility": "8 km",
    "dew_point": "21 °C",
    "feels_like": "41 °C"
}


**Additional Exercise #3: Forecast Data Visualizer (Animated)**

Functionality: Build a command-line Python program that reads extended weather forecast data from a CSV file generated using timeanddate.com/weather/philippines and produces an animated line chart showing temperature trends over time for any Philippine city.

Input:

• City name (string), entered by the user interactively.

• Number of forecast days to visualize (integer ≤ 7), entered by the user; defaults to 3 if no valid input is provided.

Calculations:

• Read weather_data.csv and filter records for the specified city.

• Extract forecast_date, temperature_min, and temperature_max.

• Use matplotlib.animation to animate a line chart showing how temperatures change over the forecast period.

• Optionally annotate each point with corresponding humidity or condition values.

• Label axes and title the chart with the city name and selected date range.

• Export the animation as a GIF or MP4 file.

Output:

• An animated chart file named forecast_plot.gif (or forecast_plot.mp4) that visualizes the temperature trends over time for the selected city, based on data including extended weather attributes.

In [4]:



# ─────────────────────────────────────────────────────────────────────────────
# ┌─────────────────────────────────────────────────────────────────────────┐
# │  CELL 4 — Additional Exercise 3: Forecast Data Visualizer (Animated)   │
# └─────────────────────────────────────────────────────────────────────────┘
#
#  INPUT GUIDE
#  -----------
#  "Enter city name to visualize"
#      Must match a city already in weather_data.csv (run Exercise 1 first).
#      Press ENTER (blank) to use the first city found in the file.
#
#  "Number of forecast days to visualize (1-7)"
#      Whole number 1-7, e.g.:  5
#      Press ENTER (blank) for default: 3
#
#  OUTPUT: forecast_plot.gif saved in the working directory.
#          Open it with any browser or image viewer to watch the animation.
# ─────────────────────────────────────────────────────────────────────────────

def _to_float(s):
    """Extract first numeric value from a string like '28 C' or '28.5'."""
    m = re.search(r"-?\d+\.?\d*", str(s))
    return float(m.group()) if m else None


def _render_frame(ax, fig, canvas, x_pos, t_min, t_max, humidity, labels, frame,
                  CLR_MAX, CLR_MIN, CLR_MUTE, CLR_TEXT, CLR_GRID, y_pad):
    """Draw one animation frame onto the figure and return a Pillow Image."""
    import io
    from PIL import Image

    ax.cla()   # clear axes completely so each frame is a clean redraw

    # Restore static axes formatting
    BG_PANEL = "#161b22"
    ax.set_facecolor(BG_PANEL)
    for spine in ax.spines.values():
        spine.set_edgecolor(CLR_GRID)
    ax.tick_params(colors=CLR_TEXT, labelsize=9)
    ax.set_xlim(-0.5, len(x_pos) - 0.5)
    ax.set_ylim(min(t_min) - y_pad, max(t_max) + y_pad + 2)
    ax.set_xticks(x_pos)
    ax.set_xticklabels(labels, rotation=22, ha="right", color=CLR_TEXT, fontsize=8.5)
    ax.set_xlabel("Forecast Date", color=CLR_MUTE, labelpad=8)
    ax.set_ylabel("Temperature (C)", color=CLR_MUTE, labelpad=8)
    ax.grid(axis="y", color="#21262d", linestyle="--", linewidth=0.9, alpha=0.8)
    ax.grid(axis="x", color="#21262d", linestyle=":",  linewidth=0.6, alpha=0.5)

    xs = x_pos[:frame + 1]

    # Shaded band between min and max
    ax.fill_between(xs, t_min[:frame + 1], t_max[:frame + 1],
                    alpha=0.15, color=CLR_MAX, zorder=1)

    # Lines
    ax.plot(xs, t_max[:frame + 1], color=CLR_MAX, linewidth=2.5,
            marker="o", markersize=8, label="Max Temp (C)", zorder=3)
    ax.plot(xs, t_min[:frame + 1], color=CLR_MIN, linewidth=2.5,
            marker="o", markersize=8, label="Min Temp (C)", zorder=3)

    # Annotate the latest point
    ax.annotate(f"{t_max[frame]:.0f}C",
                xy=(xs[-1], t_max[frame]),
                xytext=(6, 6), textcoords="offset points",
                color=CLR_MAX, fontsize=9, fontweight="bold")
    ax.annotate(f"{t_min[frame]:.0f}C",
                xy=(xs[-1], t_min[frame]),
                xytext=(6, -16), textcoords="offset points",
                color=CLR_MIN, fontsize=9, fontweight="bold")

    # Humidity annotation
    hum = humidity[frame] if frame < len(humidity) else "N/A"
    if hum and hum != "N/A":
        ax.annotate(f"Hum: {hum}",
                    xy=(xs[-1], min(t_min) - y_pad + 0.3),
                    ha="center", color=CLR_MUTE, fontsize=7.5)

    # Status text
    ax.text(0.01, 0.97,
            f"Day {frame+1}/{len(x_pos)}  |  hi {t_max[frame]:.0f}  "
            f"lo {t_min[frame]:.0f}  hum {hum}",
            transform=ax.transAxes, color=CLR_MUTE, fontsize=8, va="top")

    ax.legend(facecolor="#21262d", edgecolor=CLR_GRID,
              labelcolor=CLR_TEXT, fontsize=9, loc="upper right")

    # Capture frame as PIL Image via in-memory PNG
    canvas.draw()
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=120, facecolor=fig.get_facecolor())
    buf.seek(0)
    return Image.open(buf).copy()   # .copy() detaches from the buffer


def exercise3_forecast_visualizer():
    print("=" * 62)
    print("  EXERCISE 3 - Forecast Data Visualizer  ->  Animated GIF")
    print("=" * 62)

    CSV_PATH = "weather_data.csv"
    if not os.path.exists(CSV_PATH):
        print(f"\nERROR: '{CSV_PATH}' not found.")
        print("Please run CELL 2 (Exercise 1) first to generate the data.")
        return

    df = pd.read_csv(CSV_PATH, encoding="utf-8")
    if df.empty:
        print(f"ERROR: '{CSV_PATH}' is empty. Run Exercise 1 first.")
        return

    available_cities = df["city"].dropna().unique().tolist()
    print(f"\nCities in '{CSV_PATH}': {', '.join(available_cities)}")

    raw_city = input(
        f"\nEnter city name to visualize\n"
        f"[press ENTER for default: {available_cities[0]}] > "
    ).strip().lower().replace(" ", "-")
    city_slug = raw_city or available_cities[0]

    try:
        vis_days = int(
            input("Number of forecast days to visualize (1-7) [press ENTER for default: 3] > ").strip()
        )
        vis_days = max(1, min(7, vis_days))
    except (ValueError, EOFError):
        vis_days = 3
        print("  Using default: 3 days.")

    # Filter & clean data
    city_df = df[df["city"] == city_slug].copy()
    if city_df.empty:
        print(f"\nERROR: No data found for city '{city_slug}'.")
        print(f"Available cities: {', '.join(available_cities)}")
        return

    city_df["t_min"] = city_df["temperature_min"].apply(_to_float)
    city_df["t_max"] = city_df["temperature_max"].apply(_to_float)
    city_df = city_df.dropna(subset=["t_min", "t_max"])
    city_df = city_df.drop_duplicates(subset=["forecast_date"]).head(vis_days).reset_index(drop=True)

    if city_df.empty:
        print("\nERROR: No numeric temperature data found after cleaning.")
        print("Make sure Exercise 1 ran and logged min/max temperatures.")
        return

    labels   = city_df["forecast_date"].tolist()
    t_min    = city_df["t_min"].tolist()
    t_max    = city_df["t_max"].tolist()
    humidity = city_df["humidity"].tolist()
    x_pos    = list(range(len(labels)))

    # Colours
    CLR_MAX  = "#ff7b72"
    CLR_MIN  = "#79c0ff"
    CLR_GRID = "#30363d"
    CLR_TEXT = "#e6edf3"
    CLR_MUTE = "#8b949e"
    BG_DARK  = "#0d1117"
    BG_PANEL = "#161b22"
    y_pad    = 3

    # Build figure using the Agg backend explicitly via FigureCanvasAgg
    # This guarantees GIF output regardless of Jupyter's display backend.
    from matplotlib.backends.backend_agg import FigureCanvasAgg
    from matplotlib.figure import Figure
    from PIL import Image

    fig = Figure(figsize=(11, 5.5))
    fig.patch.set_facecolor(BG_DARK)
    canvas = FigureCanvasAgg(fig)
    ax = fig.add_subplot(111)

    city_title = city_slug.replace("-", " ").title()
    fig.suptitle(
        f"Weather Forecast  -  {city_title}  "
        f"({vis_days} day{'s' if vis_days > 1 else ''})",
        fontsize=13, fontweight="bold", color=CLR_TEXT, y=0.98
    )
    fig.tight_layout(rect=[0, 0, 1, 0.95])

    # Render each frame and collect PIL images
    print(f"\nRendering {len(x_pos)} frame(s)...")
    frames_pil = []
    for frame in range(len(x_pos)):
        img = _render_frame(
            ax, fig, canvas, x_pos, t_min, t_max, humidity, labels, frame,
            CLR_MAX, CLR_MIN, CLR_MUTE, CLR_TEXT, CLR_GRID, y_pad
        )
        frames_pil.append(img)
        print(f"  Frame {frame + 1}/{len(x_pos)} done")

    plt.close("all")

    # Save as animated GIF using Pillow directly — guaranteed to work
    GIF_PATH = "forecast_plot.gif"
    frames_pil[0].save(
        GIF_PATH,
        save_all=True,
        append_images=frames_pil[1:],
        duration=850,       # ms per frame
        loop=0,             # 0 = loop forever
        optimize=False,
    )

    print(f"\nAnimation saved to '{GIF_PATH}'")
    print(f"Open '{GIF_PATH}' in your browser or Windows Photos to watch it play.")

    # Summary table
    print(f"\nData visualized for '{city_slug}':")
    print(f"  {'Date':<30}  {'Min(C)':>7}  {'Max(C)':>7}  {'Humidity':>10}")
    print("  " + "-" * 58)
    for i, lbl in enumerate(labels):
        hum = humidity[i] if i < len(humidity) else "N/A"
        print(f"  {lbl:<30}  {t_min[i]:>7.1f}  {t_max[i]:>7.1f}  {hum:>10}")


exercise3_forecast_visualizer()

  EXERCISE 3 - Forecast Data Visualizer  ->  Animated GIF

Cities in 'weather_data.csv': Manila, manila


  Using default: 3 days.

Rendering 3 frame(s)...
  Frame 1/3 done
  Frame 2/3 done
  Frame 3/3 done

Animation saved to 'forecast_plot.gif'
Open 'forecast_plot.gif' in your browser or Windows Photos to watch it play.

Data visualized for 'Manila':
  Date                             Min(C)   Max(C)    Humidity
  ----------------------------------------------------------
  2026-04-27                         26.0     32.0         69%
  2026-04-28                         26.0     32.0         69%
  2026-04-29                         26.0     32.0         69%


# THE END OF CRESENCIO'S LAB 9 ADDITIONAL EXERCISE